# Why did this answer block a release?

Run the cells from the lab folder. Compare two refund answers against one policy, then repair the one that fails.

The policy says **5 business days after approval**. The answer says **14 business days**. In the supplied example, the program finds no support for 14 and blocks the release (`HOLD`). Try the change below and compare the two results.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

lab = Path.cwd() / "assertion_gate.py"
samples = lab.parent / "samples" / "claims.jsonl"
command = [sys.executable, str(lab), "--lang", "en"]
before = subprocess.run(command, check=True, capture_output=True, text=True).stdout
print(before, end="")
assert "-> HOLD" in before

Before opening the next cell, write down the number you would use and why. Predict what the evaluator will still miss even after the correction.

### Compare with the worked solution

Set the deadline to **5**. The evaluator should allow this case (`PROMOTE`). The condition **after approval** is still absent from the answer.

In [ ]:
rows = [json.loads(line) for line in samples.read_text(encoding="utf-8").splitlines()]
failing_case = next(row for row in rows if row["id"] == "refund-14d")
failing_case["claim"] = failing_case["claim"].replace("14", "5")

with TemporaryDirectory(prefix="lumina-lab-") as folder:
    candidate = Path(folder) / "claims.jsonl"
    candidate.write_text("\n".join(json.dumps(row) for row in rows) + "\n", encoding="utf-8")
    after = subprocess.run(
        [*command, "--input", str(candidate), "--ci"], check=True, capture_output=True, text=True
    ).stdout

print(after, end="")
assert "-> PROMOTE" in after

The lexical check can miss a negation or the approval condition. Which additional check would you ask for before releasing this answer?